# Real LLM Generation & Inference

The goal of is to move from the conceptual LLM pipeline established in Day 1 to real model inference.

This day focuses on:

- Real model execution
- Real logits inspection
- Logits-to-probabilities conversion
- Next-token candidate selection
- Greedy decoding
- Sampling
- Temperature
- Top-k and Top-p sampling
- Seeded reproducibility
- Autoregressive generation
- Stop conditions
- Token-level generation tracing
- Input and output token accounting
- Generation latency and throughput
- Prefill and decode behavior
- KV cache concepts and inspection
- Performance benchmarking
- Engineering architecture for the generation pipeline

The main inference workflow is:

```text
Real Prompt
    ↓
Real Tokenizer
    ↓
Real Model Forward Pass
    ↓
Real Logits
    ↓
Probabilities
    ↓
Decoding Strategy
    ↓
Selected Token
    ↓
Append Token
    ↓
Repeat
    ↓
Generated Response
```

The second part of the day examines runtime behavior:

```text
Prompt Processing
      ↓
    Prefill
      ↓
   KV Cache
      ↓
    Decode
      ↓
Token-by-token Generation
```

# 2. Environment Check

The environment is verified before starting real inference experiments.

The baseline includes:

- Python 3.13
- PyTorch with CUDA support
- CUDA availability
- NVIDIA GPU
- Transformers
- Tokenizers
- Ollama
- Local Qwen model

The purpose of this check is to confirm that the notebook is using the intended runtime and GPU-enabled Python environment.

In [7]:
# Verify the Runtime environment.

import sys
import torch
import transformers
import tokenizers

print("Python:", sys.version.split()[0])
print("Python Executable:", sys.executable)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA Build:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA Device Count:", torch.cuda.device_count())

Python: 3.13.0
Python Executable: c:\Users\mahmu\AppData\Local\Programs\Python\Python313\python.exe
Transformers: 5.18.0
Tokenizers: 0.23.2
PyTorch: 2.12.0+cu130
CUDA Build: 13.0
CUDA Available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
CUDA Device Count: 1


### Result Interpretation

The notebook is running on Python 3.13 with PyTorch CUDA 13.0 support. CUDA is available and the RTX 5070 Laptop GPU is detected successfully.

### Finding

The Day 2 environment is ready for GPU-accelerated inference experiments.

# 3. Load the Real Model
## 3.1 Verify Ollama

Ollama provides the local runtime used to execute the installed `qwen3.8:27b` model.

This check verifies that the Ollama executable is available from the current environment before inspecting the target model.

No model download or installation is performed in this step.

In [8]:
# Verify that the Ollama executable is available.

import shutil
import subprocess

ollama_path = shutil.which("ollama")

print("Ollama Path:", ollama_path)

if ollama_path:
    result = subprocess.run(
        ["ollama", "--version"],
        capture_output=True,
        text=True,
    )

    print("Ollama Version:", result.stdout.strip())
else:
    print("Ollama is not available in PATH.")

Ollama Path: C:\Users\mahmu\AppData\Local\Programs\Ollama\ollama.EXE
Ollama Version: ollama version is 0.35.1


### Result Interpretation

The Ollama executable is available from the current Python environment, and Ollama version 0.35.1 is detected successfully.

### Finding

The Ollama runtime is ready for local model verification.

## 3.2 Verify `qwen3.8:27b`

The target local model is `qwen3.8:27b`.

This step verifies that the model is installed and available through the Ollama runtime.

No model download is performed.

In [ ]:
# Verify that the target local model is installed in Ollama.

import subprocess

MODEL_NAME = "qwen3.8:27b"

result = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True,
    check=True,
)

print("Target Model:", MODEL_NAME)
print("\nInstalled Ollama Models:\n")
print(result.stdout)

model_available = any(
    line.startswith(MODEL_NAME) for line in result.stdout.splitlines()
)

print("Model Available:", model_available)

Target Model: qwen3.8:27b

Installed Ollama Models:

NAME           ID              SIZE     MODIFIED   
qwen3.8:27b    aaee06c39dcf    17 GB    3 days ago    

Model Available: True


### Result Interpretation

The target `qwen3.8:27b` model is installed locally in Ollama with a reported size of 17 GB.

### Finding

The required local model is available and ready for runtime configuration.

## 3.3 Model and Runtime Configuration

The local inference configuration defines the model and runtime used throughout Day 2.

The baseline configuration is:

```text
Runtime: Ollama
Model: qwen3.8:27b
Generation Mode: Local
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
```

The Ollama runtime is used for end-to-end execution of the 27B model.

Low-level tensor experiments that require direct access to logits will use a compatible Transformers model interface when the Ollama API does not expose the required tensors.

In [10]:
# Define and verify the Day 2 model and runtime configuration.

MODEL_NAME = "qwen3.8:27b"
RUNTIME = "Ollama"

print("Runtime:", RUNTIME)
print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA Available:", torch.cuda.is_available())

Runtime: Ollama
Model: qwen3.8:27b
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
CUDA Available: True


### Result Interpretation

The runtime is configured to use Ollama with the local `qwen3.8:27b` model and the available RTX 5070 GPU.

### Finding

The model and runtime configuration is verified and ready for the tokenizer stage.

# 4. Load the Matching Tokenizer
## 4.1 Load Tokenizer

The tokenizer converts the input prompt into token IDs that can be processed by the language model.

The tokenizer used for the experiments is:

```text
Qwen/Qwen3.8-27B
```

The tokenizer is loaded separately from the Ollama runtime because tokenizer inspection requires direct access to the tokenizer interface.

The loaded tokenizer will be verified before it is used in subsequent inference experiments.

In [ ]:
# Load the Qwen3.8-27B tokenizer directly.

from transformers import Qwen3_5Tokenizer

TOKENIZER_NAME = "Qwen/Qwen3.8-27B"

tokenizer = Qwen3_5Tokenizer.from_pretrained(TOKENIZER_NAME)

print("Tokenizer Source:", TOKENIZER_NAME)
print("Tokenizer Class:", tokenizer.__class__.__name__)
print("Vocabulary Size:", tokenizer.vocab_size)

Tokenizer Source: Qwen/Qwen3.8-27B
Tokenizer Class: Qwen3_5Tokenizer
Vocabulary Size: 248044


### Result Interpretation

The Qwen3.8-27B tokenizer loaded successfully from the verified Hugging Face source and reports a vocabulary size of 248,044.

### Finding

The matching tokenizer interface is available for the next compatibility checks.

## 4.2 Verify Tokenizer Compatibility

The tokenizer and model configuration must be checked before performing low-level inference experiments.

The verification compares:

- Tokenizer vocabulary size
- Model vocabulary size
- Vocabulary difference

The values are reported without assuming that the two vocabulary sizes must be identical.

In [15]:
# Verify tokenizer and model configuration compatibility.

from transformers import AutoConfig

MODEL_SOURCE = "Qwen/Qwen3.8-27B"

config = AutoConfig.from_pretrained(MODEL_SOURCE)

tokenizer_vocab_size = tokenizer.vocab_size
model_vocab_size = config.text_config.vocab_size

print("Tokenizer Vocabulary Size:", tokenizer_vocab_size)
print("Model Vocabulary Size:", model_vocab_size)
print("Vocabulary Difference:", model_vocab_size - tokenizer_vocab_size)
print("Vocabulary Sizes Match:", tokenizer_vocab_size == model_vocab_size)

Tokenizer Vocabulary Size: 248044
Model Vocabulary Size: 248320
Vocabulary Difference: 276
Vocabulary Sizes Match: False


### Result Interpretation

The tokenizer reports 248,044 vocabulary entries, while the model configuration reports 248,320 entries, resulting in a difference of 276 entries.

### Finding

The tokenizer and model configuration values are verified and will be preserved as observed for the next inference experiments.

## 4.3 Verify Vocabulary Size

The tokenizer and model vocabulary sizes are explicitly verified before real inference.

The verification records both values independently because the tokenizer vocabulary size and the model configuration vocabulary size are not identical.

These verified values will be used as the reference for the following inference experiments.

In [16]:
# Verify and record the tokenizer and model vocabulary sizes.

verified_tokenizer_vocab = tokenizer.vocab_size
verified_model_vocab = config.text_config.vocab_size

print("Verified Tokenizer Vocabulary Size:", verified_tokenizer_vocab)
print("Verified Model Vocabulary Size:", verified_model_vocab)
print("Vocabulary Difference:", verified_model_vocab - verified_tokenizer_vocab)
print("Tokenizer Vocabulary Verified:", verified_tokenizer_vocab == 248044)
print("Model Vocabulary Verified:", verified_model_vocab == 248320)

Verified Tokenizer Vocabulary Size: 248044
Verified Model Vocabulary Size: 248320
Vocabulary Difference: 276
Tokenizer Vocabulary Verified: True
Model Vocabulary Verified: True


### Result Interpretation

The tokenizer and model vocabulary sizes were verified successfully. The tokenizer reports 248,044 entries and the model configuration reports 248,320 entries.

### Finding

The verified vocabulary values are recorded as the baseline for the real inference experiments.